In [1]:
import torch
import os
import json
from PIL import Image
import torch
import open_clip
from tqdm import tqdm
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================

DATASET_ROOT = "/path/to/fashion-iq"

CATEGORY = "dress"  # dress / shirt / toptee

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TOPK = 5

# =========================================================
# LOAD CLIP
# =========================================================

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(DEVICE)
model.eval()

# =========================================================
# LOAD IMAGE SPLIT
# =========================================================

split_path = os.path.join(
    DATASET_ROOT,
    "image_splits",
    f"split.{CATEGORY}.val.json"
)

with open(split_path, "r") as f:
    image_ids = json.load(f)

print(f"Loaded {len(image_ids)} images")

# =========================================================
# ENCODE ALL IMAGES
# =========================================================

all_image_features = []
all_image_paths = []

with torch.no_grad():

    for img_id in tqdm(image_ids):

        img_path = os.path.join(
            DATASET_ROOT,
            "images",
            f"{img_id}.jpg"
        )

        try:
            image = preprocess(Image.open(img_path).convert("RGB"))
        except:
            continue

        image = image.unsqueeze(0).to(DEVICE)

        feat = model.encode_image(image)
        feat = feat / feat.norm(dim=-1, keepdim=True)

        all_image_features.append(feat.cpu())
        all_image_paths.append(img_path)

image_features = torch.cat(all_image_features, dim=0)

print("Image feature matrix:", image_features.shape)

# =========================================================
# RETRIEVAL FUNCTIONS
# =========================================================

def retrieve_by_text(query, topk=5):

    with torch.no_grad():

        text = tokenizer([query]).to(DEVICE)

        text_feat = model.encode_text(text)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

    sims = (image_features @ text_feat.cpu().T).squeeze(1)

    vals, idxs = sims.topk(topk)

    results = []

    for score, idx in zip(vals, idxs):
        results.append(
            (all_image_paths[idx], score.item())
        )

    return results


def retrieve_by_image(query_image_path, topk=5):

    image = preprocess(
        Image.open(query_image_path).convert("RGB")
    )

    image = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        feat = model.encode_image(image)
        feat = feat / feat.norm(dim=-1, keepdim=True)

    sims = (image_features @ feat.cpu().T).squeeze(1)

    vals, idxs = sims.topk(topk + 1)

    results = []

    for score, idx in zip(vals, idxs):

        path = all_image_paths[idx]

        if path == query_image_path:
            continue

        results.append((path, score.item()))

    return results[:topk]


# =========================================================
# SIMPLE COMPOSED RETRIEVAL
# =========================================================
#
# CLIP baseline trick:
#
# composed = normalize(
#     image_feature + text_feature
# )
#
# Not SOTA, but useful baseline.
#
# =========================================================

def composed_retrieval(reference_image_path, modification_text, topk=5):

    image = preprocess(
        Image.open(reference_image_path).convert("RGB")
    )

    image = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        img_feat = model.encode_image(image)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        txt = tokenizer([modification_text]).to(DEVICE)

        txt_feat = model.encode_text(txt)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

        composed = img_feat + txt_feat
        composed = composed / composed.norm(dim=-1, keepdim=True)

    sims = (image_features @ composed.cpu().T).squeeze(1)

    vals, idxs = sims.topk(topk)

    results = []

    for score, idx in zip(vals, idxs):

        results.append(
            (all_image_paths[idx], score.item())
        )

    return results


# =========================================================
# VISUALIZATION
# =========================================================

def show_results(results, title="Results"):

    plt.figure(figsize=(15, 5))

    for i, (path, score) in enumerate(results):

        img = Image.open(path)

        plt.subplot(1, len(results), i + 1)
        plt.imshow(img)
        plt.title(f"{score:.3f}")
        plt.axis("off")

    plt.suptitle(title)
    plt.show()


# =========================================================
# EXAMPLES
# =========================================================

# -------- TEXT RETRIEVAL --------

query = "blue floral elegant dress"

results = retrieve_by_text(query, TOPK)

show_results(results, title=f"Text Query: {query}")

# -------- IMAGE RETRIEVAL --------

example_img = all_image_paths[0]

results = retrieve_by_image(example_img, TOPK)

show_results(results, title="Image Retrieval")

# -------- COMPOSED RETRIEVAL --------

modification = "make it more colorful"

results = composed_retrieval(
    example_img,
    modification,
    TOPK
)

show_results(
    results,
    title=f"Modification: {modification}"
)

ModuleNotFoundError: No module named 'open_clip'